In [1]:
from guppylang.std.builtins import array, nat, qubit
from guppylang import guppy
from guppylang.std.quantum import measure_array, x, h, measure, cx, cz

from hugr.hugr.render import RenderConfig

n = guppy.nat_var("n")
c = guppy.nat_var("c")

@guppy.unitary
class foo:

    @guppy
    def __call__(q1: array[qubit, 2]) -> None:
        h(q1[0])

    @guppy
    def call_daggered(q1: array[qubit, 2]) -> None:
        x(q1[0])

    @guppy
    def call_controlled(q1: array[qubit, 2], _controls: array[qubit, n]) -> None:
        cz(_controls[0], q1[0])

    @guppy
    def call_ctrl_daggered(q1: array[qubit, n], _controls: array[qubit, c]) -> None:
        cx(_controls[0], q1[0])
        

@guppy
def main() -> None:
    q1 = array(qubit(), qubit())
    foo(q1)
    measure_array(q1)

main.check()
# main.compile().modules[0] #.render_dot(RenderConfig(max_node_label_length=50, max_metadata_length=None)) #.view()


Error: Expected signature compatible with array[qubit, 2] -> None (at <In[1]>:26:4)
   | 
24 | 
25 |     @guppy
26 |     def call_ctrl_daggered(q1: array[qubit, n], _controls: array[qubit, c]) -> None:
   |     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
27 |         cx(_controls[0], q1[0])
   | ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Expected signature compatible with array[qubit, 2] -> None

Guppy compilation failed due to 1 previous error


In [ ]:
@guppy.protocol
class MyProto:
    @guppy.require
    def foo(self: "MyProto", xaaaa: int) -> str: ...

@guppy.struct(frozen=True)
class MyType:
    @guppy
    def foo(self: "MyType", t: int) -> str:
        return "something"

@guppy
def bar[M: MyProto](a: M) -> str:
    return a.foo(42)

# Internally desugared this should be equivalent to `bar`.
@guppy
def baz(a: MyProto) -> str:
    return a.foo(42)

@guppy
def main() -> None:
    mt = MyType()
    bar(mt)
    baz(mt)

main.compile()

Package(modules=[Hugr(module_root=Node(0), entrypoint=Node(1), _nodes=[NodeData(op=Module(), parent=None, metadata=NodeMetadata({'name': '__main__', 'core.used_extensions': [{'name': 'prelude', 'version': '0.2.2'}, {'name': 'arithmetic.int.types', 'version': '0.1.0'}], 'core.generator': {'name': 'guppylang (guppylang-internals-v1.0.0-a4)', 'version': '1.0.0-a4'}})), NodeData(op=FuncDefn(f_name='__main__.main', inputs=[], params=[], visibility='Public'), parent=Node(0), metadata=NodeMetadata({'tket.unitary': 0})), NodeData(op=Input(types=[]), parent=Node(1), metadata=NodeMetadata({})), NodeData(op=Output(), parent=Node(1), metadata=NodeMetadata({})), NodeData(op=CFG(inputs=[]), parent=Node(1), metadata=NodeMetadata({})), NodeData(op=DataflowBlock(inputs=[], _sum=Unit), parent=Node(4), metadata=NodeMetadata({})), NodeData(op=Input(types=[]), parent=Node(5), metadata=NodeMetadata({})), NodeData(op=Output(), parent=Node(5), metadata=NodeMetadata({})), NodeData(op=ExitBlock(), parent=Node(4

In [8]:
from collections.abc import Callable

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    Controllable,
    Daggerable,
    Unitary,
    control,
    dagger,
    owned,
    power,
    result,
)
from guppylang.std.num import nat
from guppylang.std.quantum import (
    angle,
    cx,
    discard,
    h,
    measure,
    qubit,
    rx,
    discard_array,
)
from guppylang_internals.metadata.common import (
    CONTROLLED_KEY,
    CTRL_DAGGERED_KEY,
    DAGGERED_KEY,
)
from hugr.ops import FuncDefn

def test_unitary_class_functions_are_regular_functions():
    @guppy.unitary
    class foo:
        @guppy
        def helper(q: qubit) -> None:
            h(q)

        @guppy
        def __call__(q: qubit) -> None:
            helper(q)

        @guppy
        def call_daggered(q: qubit) -> None:
            helper(q)

        @guppy
        def call_controlled(q: qubit, _controls: array[qubit, 1]) -> None:
            helper(q)
            helper(_controls[0])

        @guppy
        def call_ctrl_daggered(q: qubit, _controls: array[qubit, 1]) -> None:
            helper(q)
            helper(_controls[0])

    @guppy
    def main() -> None:
        q = qubit()
        foo(q)
        result("r", measure(q).read())

    return main

test_unitary_class_functions_are_regular_functions().emulator(n_qubits=1).with_shots(100).run().collated_counts()

Counter({(('r', '1'),): 53, (('r', '0'),): 47})

In [2]:

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    control,
    dagger,
)
from guppylang.std.num import nat
from guppylang.std.quantum import ( cx,
    h, x,
    measure,
    qubit)
import guppylang
guppylang.enable_experimental_features()

def test_unitary_class_functions_are_regular_functions():
    @guppy.unitary
    class foo:
        @guppy
        def __call__(q: qubit) -> None:
            h(q)
            c = qubit()

        @guppy
        def call_daggered(q: qubit) -> None:
            x(q)

        @guppy
        def call_controlled(q: qubit, _controls: array[qubit, 1]) -> None:
            cx(q, _controls[0])

        @guppy
        def call_ctrl_daggered(q: qubit, _controls: array[qubit, 1]) -> None:
            cx(q, _controls[0])

    @guppy
    def main() -> None:
        q = qubit()
        c = qubit()
        with control(c):
            foo(q)
        with dagger:
            foo(q)
        with control(c), dagger:
            foo(q)
        measure(q)
        measure(c)

    return main

test_unitary_class_functions_are_regular_functions().compile()

Error: Drop violation (at <In[2]>:22:12)
   | 
20 |         def __call__(q: qubit) -> None:
21 |             h(q)
22 |             c = qubit()
   |             ^ Variable `c` with non-droppable type `qubit` is leaked

Help: Make sure that `c` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error
